In [ ]:
!pip install numpy pandas matplotlib scipy

## Load all NCU data

In [ ]:
import pandas as pd

metrics_df_raw: pd.DataFrame = pd.read_csv('metrics/metrics_all_ncu.csv')
metrics_df_raw

In [ ]:
# convert stall cols to proportions instead of raw sample counts
stall_cols = [c for c in metrics_df_raw.columns if c.startswith('smsp__pcsamp_warps_issue_stalled_')]
metrics_df = metrics_df_raw.copy()
total_samples = metrics_df_raw[stall_cols].sum(axis=1)
metrics_df[stall_cols] = metrics_df_raw[stall_cols].div(total_samples, axis=0)
metrics_df

In [ ]:
metrics_df.info()

In [ ]:
LABEL_COLS = ['GpuName', 'SeqLen', 'HeadDim', 'Run', 'Action', 'Range']

## Compare runs for sanity check

In [ ]:
metrics_df_run0 = metrics_df[metrics_df['Run'] == 0].reset_index(drop=True)
metrics_df_run1 = metrics_df[metrics_df['Run'] == 1].reset_index(drop=True)
metrics_df_run2 = metrics_df[metrics_df['Run'] == 2].reset_index(drop=True)
metrics_df_run0

In [ ]:
import numpy as np
from scipy.stats import variation

data_cols = [c for c in metrics_df_run0.columns if c not in LABEL_COLS]

# Stack runs along axis 0 → shape (3, n_rows, n_data_cols)
stacked = np.stack([
    metrics_df_run0[data_cols].values,
    metrics_df_run1[data_cols].values,
    metrics_df_run2[data_cols].values
], axis=0)

# CV = std/mean across the 3 runs for each (row, metric) pair
diff_df = metrics_df_run0[LABEL_COLS].copy()
diff_df[data_cols] = variation(stacked, axis=0)

diff_df

In [ ]:
threshold = 0.03
data_cols = [c for c in diff_df.columns if c not in LABEL_COLS]

flagged = (
    diff_df[data_cols]
    .stack()
    .reset_index()
    .rename(columns={'level_1': 'metric', 0: 'cv'})
    .query('cv >= @threshold')
)

label_vals = diff_df.loc[flagged['level_0'], LABEL_COLS].reset_index(drop=True)
flagged_df = pd.concat([label_vals, flagged[['metric', 'cv']].reset_index(drop=True)], axis=1)
flagged_df

## Get Utilization Percentages

### Individual Kernels

In [ ]:
sm_cycles_elapsed = metrics_df_run0['sm__cycles_elapsed.avg']

util_df = metrics_df_run0[LABEL_COLS]
util_df['SM_Util'] = metrics_df_run0['sm__cycles_active.avg'] / sm_cycles_elapsed
util_df['Tensor_Util'] = metrics_df_run0['sm__pipe_tensor_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Util'] = metrics_df_run0['sm__pipe_fma_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Heavy_Util'] = metrics_df_run0['sm__pipe_fmaheavy_cycles_active.avg'] / sm_cycles_elapsed
util_df['FMA_Lite_Util'] = metrics_df_run0['sm__pipe_fmalite_cycles_active.avg'] / sm_cycles_elapsed
util_df['Shared_Util'] = metrics_df_run0['sm__pipe_shared_cycles_active.avg'] / sm_cycles_elapsed
util_df['ALU_Util'] = metrics_df_run0['sm__pipe_alu_cycles_active.avg'] / sm_cycles_elapsed
util_df['ALU_Heavy_Util'] = metrics_df_run0['sm__pipe_aluheavy_cycles_active.avg'] / sm_cycles_elapsed
# util_df['ALU_Lite_Util'] = metrics_df_run0['sm__pipe_alulite_cycles_active.avg'] / sm_cycles_elapsed
# util_df['FP16_Util'] = metrics_df_run0['sm__pipe_fp16_cycles_active.avg'] / sm_cycles_elapsed
util_df['FP64_Util'] = metrics_df_run0['sm__pipe_fp64_cycles_active.avg'] / sm_cycles_elapsed
util_df['L1_Cache_Util'] = metrics_df_run0['l1tex__cycles_active.avg'] / sm_cycles_elapsed
# util_df['L1_Cache_Util_Ampere+'] = metrics_df_run0['1tex__cycles_active.avg'] / sm_cycles_elapsed
util_df['LTS_Util'] = metrics_df_run0['lts__cycles_active.avg'] / sm_cycles_elapsed
util_df['DRAM_Util'] = metrics_df_run0['dram__cycles_active.avg'] / sm_cycles_elapsed

# Normalize from pct-of-peak (0–100) to fraction (0–1) to match other units
util_df['SFU_Util'] = metrics_df_run0['smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed'] / 100.0
util_df['SMSP_Util'] = metrics_df_run0['smsp__issue_active.avg.pct_of_peak_sustained_elapsed'] / 100.0

util_df

In [ ]:
util_df.to_csv('metrics/util_ncu.csv')

### Aggregate across Math backend kernels

In [33]:
exclude_prefixes = ('fmha_cutlassF_', 'flash_fwd_', 'cudnn_')
metrics_df_run0_math_only = metrics_df_run0[~metrics_df_run0['Action'].str.startswith(exclude_prefixes)].reset_index(drop=True)
metrics_df_run0_math_only.head(31)

,GpuName,SeqLen,HeadDim,Run,Range,Action,dram__cycles_active.avg,gpu__time_duration.sum,l1tex__cycles_active.avg,lts__cycles_active.avg,...,smsp__pcsamp_warps_issue_stalled_short_scoreboard,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,sm__pipe_aluheavy_cycles_active.avg,sm__pipe_fmaheavy_cycles_active.avg,sm__pipe_fmalite_cycles_active.avg
0,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7456.0,1601.592593,1757.7000,...,0.020833,629.759259,0.000000,0.000000,0.000000,29.888889,240.888889,NaN,NaN,NaN
1,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7424.0,1583.740741,1842.4625,...,0.025000,611.907407,0.000000,0.000000,0.000000,29.824074,240.888889,NaN,NaN,NaN
2,a100-sxm4-40gb,256,64,0,0,unrolled_elementwise_kernel,50.7,7392.0,1582.527778,1733.3125,...,0.096774,632.439815,0.000000,0.000000,0.000000,29.761574,240.888889,NaN,NaN,NaN
3,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,251.055556,928.3125,...,NaN,100.942130,0.000000,0.000000,0.000000,1.629630,11.111111,NaN,NaN,NaN
4,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,54.9,3616.0,243.879630,1060.4500,...,0.000000,95.706019,0.000000,0.000000,0.000000,1.629630,11.111111,NaN,NaN,NaN
5,a100-sxm4-40gb,256,64,0,0,ampere_sgemm_32x32_sliced1x4_tn,115.2,8768.0,4252.379630,3465.5625,...,0.058824,655.363426,1.777778,98.685185,0.000000,150.349537,1210.032407,NaN,NaN,NaN
6,a100-sxm4-40gb,256,64,0,0,softmax_warp_forward,212.3,5024.0,1827.851852,2249.4000,...,0.456522,343.949074,0.000000,0.000000,0.000000,251.854167,401.777778,NaN,NaN,NaN
7,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,208.9,3744.0,1007.925926,1177.5500,...,0.000000,367.655093,0.000000,0.000000,0.000000,8.888889,45.037037,NaN,NaN,NaN
8,a100-sxm4-40gb,256,64,0,0,reduce_kernel,93.4,7680.0,878.314815,1525.0125,...,0.125000,615.439815,106.946759,1.192130,125.738426,111.111111,362.710648,NaN,NaN,NaN
9,a100-sxm4-40gb,256,64,0,0,vectorized_elementwise_kernel,3.1,3008.0,13.111111,220.0625,...,NaN,0.000000,0.000000,0.000000,0.000000,0.129630,1.074074,NaN,NaN,NaN


In [ ]:
MATH_UTIL_COLS = {
    'SM_Util': 'sm__cycles_active.avg',
    'Tensor_Util': 'sm__pipe_tensor_cycles_active.avg',
    'FMA_Util': 'sm__pipe_fma_cycles_active.avg',
    'FMA_Heavy_Util': 'sm__pipe_fmaheavy_cycles_active.avg',
    'FMA_Lite_Util': 'sm__pipe_fmalite_cycles_active.avg',
    'Shared_Util': 'sm__pipe_shared_cycles_active.avg',
    'ALU_Util': 'sm__pipe_alu_cycles_active.avg',
    'ALU_Heavy_Util': 'sm__pipe_aluheavy_cycles_active.avg',
    'FP64_Util': 'sm__pipe_fp64_cycles_active.avg',
    'L1_Cache_Util': 'l1tex__cycles_active.avg',
    'LTS_Util': 'lts__cycles_active.avg',
    'DRAM_Util': 'dram__cycles_active.avg',
}

# Percent-of-peak metrics require weighted-average aggregation:
#   weighted_util = sum((U_i / 100) * elapsed_i) / sum(elapsed_i)
MATH_PCT_COLS = {
    'SFU_Util': 'smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed',
    'SMSP_Util': 'smsp__issue_active.avg.pct_of_peak_sustained_elapsed',
}

GROUP_COLS = ['GpuName', 'SeqLen', 'HeadDim']

# Pre-compute weighted products for pct-of-peak metrics before groupby
math_df = metrics_df_run0_math_only.copy()
for util_name, raw_col in MATH_PCT_COLS.items():
    math_df[f'_weighted_{util_name}'] = math_df[raw_col] * math_df['sm__cycles_elapsed.avg']

weighted_cols = [f'_weighted_{k}' for k in MATH_PCT_COLS]
agg_cols = list(MATH_UTIL_COLS.values()) + ['sm__cycles_elapsed.avg'] + weighted_cols

math_agg = (
    math_df[GROUP_COLS + agg_cols]
    .groupby(GROUP_COLS)
    .sum()
    .reset_index()
)

util_df_math_only = math_agg[GROUP_COLS].copy()
for util_col, raw_col in MATH_UTIL_COLS.items():
    util_df_math_only[util_col] = math_agg[raw_col] / math_agg['sm__cycles_elapsed.avg']

for util_name in MATH_PCT_COLS:
    util_df_math_only[util_name] = (
            math_agg[f'_weighted_{util_name}'] / math_agg['sm__cycles_elapsed.avg'] / 100
    )

util_df_math_only

In [ ]:
util_df_math_only.to_csv('metrics/util_ncu_math.csv')

### Aggregate across FlashAttention backend kernels

In [34]:
metrics_df_run0_flash_only = metrics_df_run0[metrics_df_run0['Action'].str.startswith('flash_fwd_')].reset_index(drop=True)
metrics_df_run0_flash_only.head()

,GpuName,SeqLen,HeadDim,Run,Range,Action,dram__cycles_active.avg,gpu__time_duration.sum,l1tex__cycles_active.avg,lts__cycles_active.avg,...,smsp__pcsamp_warps_issue_stalled_short_scoreboard,smsp__warps_issue_stalled_long_scoreboard.avg,smsp__warps_issue_stalled_math_pipe_throttle.avg,smsp__warps_issue_stalled_mio_throttle.avg,smsp__warps_issue_stalled_not_selected.avg,smsp__warps_issue_stalled_short_scoreboard.avg,smsp__warps_issue_stalled_wait.avg,sm__pipe_aluheavy_cycles_active.avg,sm__pipe_fmaheavy_cycles_active.avg,sm__pipe_fmalite_cycles_active.avg
0,a100-sxm4-40gb,256,64,0,0,flash_fwd_kernel,128.0,16320.0,293.259259,2105.3875,...,0.250000,18.168981,0.000000,2.699074,0.0,11.791667,78.451389,NaN,NaN,NaN
1,a100-sxm4-40gb,256,128,0,0,flash_fwd_splitkv_kernel,194.8,11296.0,748.250000,2577.7375,...,0.250000,74.277778,0.296296,16.314815,0.0,44.745370,175.550926,NaN,NaN,NaN
2,a100-sxm4-40gb,256,128,0,0,flash_fwd_splitkv_combine_kernel,224.3,8320.0,3723.925926,2295.6000,...,0.254237,715.375000,1.777778,0.000000,0.0,315.407407,809.909722,NaN,NaN,NaN
3,a100-sxm4-40gb,512,64,0,0,flash_fwd_splitkv_kernel,206.2,12064.0,1607.925926,2583.0750,...,0.250000,149.388889,0.592593,41.555556,0.0,67.275463,334.743056,NaN,NaN,NaN
4,a100-sxm4-40gb,512,64,0,0,flash_fwd_splitkv_combine_kernel,225.7,8224.0,3743.972222,2375.3500,...,0.359375,742.125000,1.777778,0.000000,0.0,315.407407,809.916667,NaN,NaN,NaN


## Plot Setup

In [ ]:
import os
import matplotlib.pyplot as plt

In [ ]:
UNIT_COLS = sorted([col for col in util_df.columns if col.endswith('_Util')])
UNIT_COLS

In [ ]:
UNIT_LABELS = [col[:col.index('_Util')] for col in UNIT_COLS]
UNIT_LABELS

In [ ]:
GPU_LABELS = {
    'rtx2060': 'RTX 2060',
    't4': 'T4',
    'a100-sxm4-40gb': 'A100',
    'l4': 'L4',
    'h10080gbhbm3': 'H100',
    'rtxpro6000blackwellserveredition': 'Blackwell',
}
GPU_ORDER = ['RTX 2060', 'T4', 'A100', 'L4', 'H100', 'Blackwell']

## Plot Relative Durations of Kernels in Math Backend

In [ ]:
def plot_math_kernel_durations(gpu_name: str, seq_len: int, head_dim: int,
                               show: bool = False, save: bool = False):
    """
    Bar chart of total sm__cycles_elapsed.avg per Math-backend kernel name
    for a single (gpu_name, seq_len, head_dim) config.

    Excludes fmha_cutlassF_ (xformers/MEA), flash_fwd_ (FA2), and cudnn_ kernels.
    Multiple invocations of the same kernel name are summed.
    """
    reverse_gpu_labels = {v: k for k, v in GPU_LABELS.items()}
    raw_gpu_name = reverse_gpu_labels[gpu_name]

    sub = metrics_df_run0_math_only[
        (metrics_df_run0_math_only['GpuName'] == raw_gpu_name) &
        (metrics_df_run0_math_only['SeqLen'] == seq_len) &
        (metrics_df_run0_math_only['HeadDim'] == head_dim)
        ].copy()

    assert not sub.empty, (
        f"No Math kernel data for gpu_name='{gpu_name}', seq_len={seq_len}, head_dim={head_dim}"
    )

    agg = (sub.groupby('Action')['sm__cycles_elapsed.avg']
           .sum()
           .sort_values(ascending=False))

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(agg)), agg.values)
    ax.set_xticks(range(len(agg)))
    ax.set_xticklabels(agg.index, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Math Kernel Durations — {gpu_name}, SeqLen={seq_len}, HeadDim={head_dim}')
    ax.set_xlabel('Kernel')
    ax.set_ylabel('Total sm__cycles_elapsed.avg (cycles)')
    fig.tight_layout()

    if save:
        os.makedirs('plots/math_kernel_durations', exist_ok=True)
        plt.savefig(f'plots/math_kernel_durations/{gpu_name}_{seq_len}x{head_dim}.png')

    if show:
        plt.show()
    else:
        plt.close()

In [ ]:
# Example
plot_math_kernel_durations('A100', 2048, 128, show=True)

In [ ]:
for gpu_name in GPU_ORDER:
    for seq_len in metrics_df_run0['SeqLen'].unique():
        for head_dim in metrics_df_run0['HeadDim'].unique():
            try:
                plot_math_kernel_durations(gpu_name, seq_len, head_dim, save=True)
            except Exception as e:
                print(e)

Note that `volta_sgemm_*`, `ampere_sgemm_*`, `sm80_xmma_gemm_`, and `Kernel2` kernels are all Matrix Mult

## Plot Utilization of Function Units

The following sections exclude the Math backend.

### Setup

In [ ]:
KERNEL_ORDER = ['xformers', 'FA2', 'cuDNN']


def label_kernel(name: str):
    """
    Currently excludes Math kernels
    """
    if 'flash_fwd_splitkv_combine' in name: return None  # exclude reduction step
    if 'flash_fwd' in name: return 'FA2'  # covers flash_fwd_kernel and split_kv
    if 'cudnn_generated' in name: return 'cuDNN'
    if 'fmha_cutlassF' in name: return 'xformers'
    return None

In [ ]:
# Filter and label
plot_df = util_df.copy()
plot_df['KernelLabel'] = plot_df['Action'].apply(label_kernel)
plot_df['GpuLabel'] = plot_df['GpuName'].map(GPU_LABELS)

plot_df = plot_df[
    plot_df['KernelLabel'].notna() &
    plot_df['GpuLabel'].isin(GPU_ORDER)
    ].copy()

# Average over repeated NCU invocations for the same (GPU, kernel, input) config
# If only using one run in util_df, only one of each anyway
agg_df = (plot_df
          .groupby(['GpuLabel', 'KernelLabel', 'SeqLen', 'HeadDim'])[UNIT_COLS]
          .mean()
          .reset_index())

print(f"agg_df shape: {agg_df.shape}")
print(agg_df.groupby(['KernelLabel', 'GpuLabel']).size().unstack(fill_value=0))


In [ ]:
SEQ_LENS = agg_df['SeqLen'].unique()
HEAD_DIMS = agg_df['HeadDim'].unique()

### Plot util of each Function Unit across GPU and Kernel for certain Seq Len, Head Dim

In [ ]:
def plot_util_by_gpu_and_kernel(seq_len: int, head_dim: int, util_col: str, show: bool = False,
                                save: bool = False):
    """
    Bar chart of util_col across kernels and GPUs for a single (seq_len, head_dim) config.
    """
    sub = agg_df[(agg_df['SeqLen'] == seq_len) & (agg_df['HeadDim'] == head_dim)]
    pivot = (sub
             .pivot_table(index='KernelLabel', columns='GpuLabel', values=util_col)
             .reindex(index=KERNEL_ORDER, columns=GPU_ORDER))

    col_label = dict(zip(UNIT_COLS, UNIT_LABELS)).get(util_col, util_col)

    fig, ax = plt.subplots(figsize=(7, 4))
    pivot.plot(kind='bar', ax=ax, width=0.75, rot=25)
    ax.set_title(f'{col_label} Utilization \u2014 SeqLen={seq_len}, HeadDim={head_dim}')
    ax.set_xlabel('')
    ax.set_ylabel(f'{col_label} Utilization (fraction of cycles)')
    ax.set_ylim(0, 1)
    ax.legend(title='GPU', fontsize=8)
    fig.tight_layout()

    if not os.path.exists(f'plots'):
        os.makedirs(f'plots')
    if not os.path.exists(f'plots/util_across_gpu_and_kernel'):
        os.makedirs(f'plots/util_across_gpu_and_kernel')
    if not os.path.exists(f'plots/util_across_gpu_and_kernel/{util_col}'):
        os.makedirs(f'plots/util_across_gpu_and_kernel/{util_col}')

    if save:
        plt.savefig(f'plots/util_across_gpu_and_kernel/{util_col}/{util_col}_{seq_len}x{head_dim}.png')

    if show:
        plt.show()
    else:
        plt.close()


In [ ]:
# Example
plot_util_by_gpu_and_kernel(256, 128, 'Tensor_Util', show=True)

In [ ]:
for unit_col in UNIT_COLS:
    print(f"Plotting {unit_col}")
    for seq_len in SEQ_LENS:
        print(f"  Seq len {seq_len}")
        for head_dim in HEAD_DIMS:
            print(f"    Head dim {head_dim}")
            plot_util_by_gpu_and_kernel(seq_len, head_dim, unit_col, save=True)

### Plot util of all Function Units for certain GPU, Kernel, Seq Len, Head Dim

In [ ]:
def plot_util_all_func_units(gpu_name: str, kernel_name: str, seq_len: int, head_dim: int,
                             show: bool = False, save: bool = False):
    """
    Bar chart of all util_cols for a single (GPU, kernel, seq_len, head_dim) config.
    """
    sub = agg_df[
        (agg_df['GpuLabel'] == gpu_name) &
        (agg_df['KernelLabel'] == kernel_name) &
        (agg_df['SeqLen'] == seq_len) &
        (agg_df['HeadDim'] == head_dim)
        ]

    assert not sub.empty, (
        f"No data for gpu_name='{gpu_name}', kernel_name='{kernel_name}', "
        f"seq_len={seq_len}, head_dim={head_dim}"
    )

    values = sub[UNIT_COLS].iloc[0]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(UNIT_LABELS, values)
    ax.set_title(f'Functional Unit Utilization — {gpu_name}, {kernel_name}, SeqLen={seq_len}, HeadDim={head_dim}')
    ax.set_xlabel('Functional Unit')
    ax.set_ylabel('Utilization (fraction of cycles)')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=30)
    fig.tight_layout()

    if not os.path.exists(f'plots'):
        os.makedirs(f'plots')
    if not os.path.exists(f'plots/util_all_func_units'):
        os.makedirs(f'plots/util_all_func_units')
    if not os.path.exists(f'plots/util_all_func_units/{gpu_name}'):
        os.makedirs(f'plots/util_all_func_units/{gpu_name}')

    if save:
        plt.savefig(f'plots/util_all_func_units/{gpu_name}/{gpu_name}_{kernel_name}_{seq_len}x{head_dim}.png')

    if show:
        plt.show()
    else:
        plt.close()


In [ ]:
# Example
plot_util_all_func_units('A100', 'FA2', 2048, 128, show=True)

In [ ]:
for gpu_name in GPU_ORDER:
    print(f"Plotting {gpu_name}")
    for kernel_name in KERNEL_ORDER:
        print(f"  Kernel {kernel_name}")
        for seq_len in SEQ_LENS:
            print(f"    Seq len {seq_len}")
            for head_dim in HEAD_DIMS:
                print(f"      Head dim {head_dim}")
                try:
                    plot_util_all_func_units(gpu_name, kernel_name, seq_len, head_dim, save=True)
                except AssertionError as e:
                    print(f'      {e}')